<a href="https://colab.research.google.com/github/msaleem-aisci/deep-learning/blob/main/RNN_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np

In [4]:
class Tanh:
    def forward(self, z):
        return np.tanh(z)

    def backward(self, delta, z):
        return delta * (1 - np.tanh(z)**2)

class Sigmoid:
    def forward(self, z):
        return 1 / (1 + np.exp(-z))

    def backward(self, delta, z):
        sig = self.forward(z)
        return delta * (sig * (1 - sig))

In [5]:
class Dense:
    def __init__(self, units, input_dim, activation='sigmoid'):
        self.units = units
        self.weights = np.random.randn(units, input_dim) * 0.01
        self.bias = np.zeros((units, 1))
        self.act_fn_name = activation
        self.sigmoid = Sigmoid()

    def forward(self, X):
        self.inputs = X
        self.z = np.dot(self.weights, X) + self.bias
        if self.act_fn_name == 'sigmoid':
            self.output = self.sigmoid.forward(self.z)
        else:
            self.output = self.z
        return self.output

    def backward(self, delta, lr):
        if self.act_fn_name == 'sigmoid':
            delta = self.sigmoid.backward(delta, self.z)

        w_grad = np.dot(delta, self.inputs.T)
        b_grad = np.sum(delta, axis=1, keepdims=True)
        dX = np.dot(self.weights.T, delta)

        self.weights -= lr * w_grad
        self.bias -= lr * b_grad

        return dX

In [6]:
class RNN:
    def __init__(self, units, input_dim):
        self.units = units
        self.input_dim = input_dim

        self.weights = np.random.randn(units, input_dim) * 0.01
        self.hidden_weights = np.random.randn(units, units) * 0.01
        self.bias = np.zeros((units, 1))

        self.act_fn = Tanh()
        self.cache = []

    def forward(self, X):
        # Reset hidden state for new sequence
        self.hidden_state = np.zeros((self.units, 1))
        self.cache = [] # Clear cache

        # X shape is (Sequence_Length, Input_Dim) -> (3, 1)
        for t_input in X:
            t_input = t_input.reshape(-1, 1) # Ensure shape (1, 1)

            # Save previous hidden state before update (needed for BPTT)
            h_prev = self.hidden_state.copy()

            # RNN Calculation
            # h_t = tanh(Wx * x + Wh * h_prev + b)
            z = np.dot(self.weights, t_input) + np.dot(self.hidden_weights, h_prev) + self.bias
            self.hidden_state = self.act_fn.forward(z)

            # Store in cache
            self.cache.append({
                "z": z,
                "h_next": self.hidden_state,
                "h_prev": h_prev,
                "x": t_input
            })

        # We return only the final hidden state (Many-to-One architecture)
        return self.hidden_state

    def backward(self, delta, lr):
        # Initialize gradients accumulator
        dW = np.zeros_like(self.weights)
        dWh = np.zeros_like(self.hidden_weights)
        db = np.zeros_like(self.bias)

        # 'delta' currently comes from the Dense layer (loss gradient w.r.t final hidden state)

        # Loop BACKWARDS through time
        for step_info in reversed(self.cache):
            # 1. Backprop through Tanh Activation
            # dz = delta * (1 - tanh^2(z))
            dz = self.act_fn.backward(delta, step_info['z'])

            # 2. Accumulate Gradients for Weights
            dW += np.dot(dz, step_info['x'].T)         # Input weights
            dWh += np.dot(dz, step_info['h_prev'].T)   # Hidden weights
            db += dz                                   # Bias

            # 3. CRITICAL FIX: Pass error to previous hidden state
            # This is "Time Travel". Error flows via Hidden Weights to t-1
            delta = np.dot(self.hidden_weights.T, dz)

        # 4. Gradient Clipping (prevents exploding gradients)
        for grad in [dW, dWh, db]:
            np.clip(grad, -1, 1, out=grad)

        # 5. Update Weights
        self.weights -= lr * dW
        self.hidden_weights -= lr * dWh
        self.bias -= lr * db

        return delta

In [7]:
class ANN:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, loss_grad, lr):
        grad = loss_grad
        for layer in reversed(self.layers):
            grad = layer.backward(grad, lr)

    def compute_loss(self, y, y_hat):
        # Binary Cross Entropy
        m = y.shape[0]
        loss = -(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))
        return np.mean(loss)

    def train(self, X, y, epochs=1000, lr=0.1):
        for epoch in range(epochs):
            total_loss = 0

            # Stochastic Gradient Descent (one sample at a time)
            for i in range(len(X)):
                # 1. Forward
                y_hat = self.forward(X[i])

                # 2. Loss & Gradient
                loss = self.compute_loss(y[i], y_hat)
                total_loss += loss

                # Gradient of Loss w.r.t Output (y_hat - y)
                loss_grad = (y_hat - y[i])

                # 3. Backward
                self.backward(loss_grad, lr)

            if epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {total_loss/len(X):.4f}")

In [9]:
X = np.array([
    [[1], [0], [1]],  # caffeine, no workout, phone => no sleep
    [[0], [1], [0]],  # no caffeine, workout, no phone => good sleep
    [[1], [1], [1]],  # all stimulants => no sleep
    [[0], [0], [0]],  # calm day => good sleep
    [[1], [0], [0]],  # only caffeine => maybe no sleep
])
y = np.array([[0], [1], [0], [1], [0]])

In [10]:
model = ANN([
    RNN(units=4, input_dim=1),
    Dense(units=1, input_dim=4, activation='sigmoid')
])

print("Starting Training...")
model.train(X, y, epochs=500, lr=0.1)

Starting Training...
Epoch 0, Loss: 0.6956
Epoch 100, Loss: 0.6728
Epoch 200, Loss: 0.5937
Epoch 300, Loss: 0.1013
Epoch 400, Loss: 0.0498
